# Notebook 02: Strings — The Foundation of Redis

Strings are the **simplest and most common** Redis data type. In fact, every Redis key is a string, and the most basic value type is also a string.

But don't let the name fool you — Redis strings can hold:
- Text: `"Hello World"`
- Numbers: `"42"` (Redis treats them as integers for math!)
- Serialized JSON: `'{"name": "Sujit"}'`
- Binary data: images, files (up to **512 MB**)

**What you'll learn:** SET options, expiry, conditional sets, counters, substrings, bit operations, and real-world patterns.

In [ ]:
import redis
import json
import time

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
r.flushdb()  # Start with a clean slate
print(f"Connected! Keys in DB: {r.dbsize()}")

---
## 1. SET with Options

SET is more powerful than you might think. It has several useful options:

In [ ]:
# Basic SET (you know this from Notebook 01)
# Redis CLI: SET name "Sujit"
r.set('name', 'Sujit')
print(f"name = {r.get('name')}")

### SET with Expiry (EX / PX)

You can set a key that **automatically deletes itself** after a certain time. This is incredibly useful for caching and sessions.

In [ ]:
# Redis CLI: SET session:user1 "logged_in" EX 10
# This key will auto-delete after 10 seconds
r.set('session:user1', 'logged_in', ex=10)  # ex = seconds
print(f"Session value: {r.get('session:user1')}")
print(f"Time to live: {r.ttl('session:user1')} seconds")

# You can also use milliseconds
# Redis CLI: SET flash:msg "Sale!" PX 5000
r.set('flash:msg', 'Flash sale ending soon!', px=5000)  # px = milliseconds
print(f"Flash message TTL: {r.pttl('flash:msg')} milliseconds")

In [ ]:
# Let's watch a key expire in real-time!
r.set('countdown', 'I will disappear!', ex=5)

for i in range(7):
    value = r.get('countdown')
    ttl = r.ttl('countdown')
    print(f"  Second {i}: value={value}, TTL={ttl}")
    time.sleep(1)

print("\nThe key expired and was automatically deleted!")

### SET Conditionally (NX / XX)

- `NX` = **N**ot e**X**ists — only SET if the key does **NOT** already exist
- `XX` = e**X**ists e**X**ists — only SET if the key **ALREADY** exists

In [ ]:
# NX — Only set if key doesn't exist (great for distributed locks!)
# Redis CLI: SET lock:resource1 "owner1" NX

r.delete('lock:resource1')  # Make sure it doesn't exist

# First attempt — key doesn't exist, so it works
result1 = r.set('lock:resource1', 'process_A', nx=True)
print(f"First lock attempt: {result1}")   # True — got the lock!

# Second attempt — key already exists, so it fails
result2 = r.set('lock:resource1', 'process_B', nx=True)
print(f"Second lock attempt: {result2}")  # None — lock already taken!

print(f"Lock owner: {r.get('lock:resource1')}")  # Still process_A

In [ ]:
# XX — Only set if key ALREADY exists (update, don't create)
# Redis CLI: SET config:theme "dark" XX

# This fails because the key doesn't exist yet
result1 = r.set('config:theme', 'dark', xx=True)
print(f"Update non-existent key: {result1}")  # None

# Create the key first
r.set('config:theme', 'light')

# Now XX works because the key exists
result2 = r.set('config:theme', 'dark', xx=True)
print(f"Update existing key: {result2}")  # True
print(f"Theme: {r.get('config:theme')}")  # dark

### Combining Options
You can combine NX/XX with EX/PX. This is the basis of **distributed locks**:

In [ ]:
# Distributed lock that auto-expires after 30 seconds
# Redis CLI: SET lock:job "worker1" NX EX 30
acquired = r.set('lock:job', 'worker_1', nx=True, ex=30)
if acquired:
    print("Lock acquired! Doing work...")
    # ... do your work here ...
    r.delete('lock:job')  # Release the lock
    print("Lock released.")
else:
    print("Someone else has the lock, try again later.")

---
## 2. GETSET, GETDEL, GETEX

In [ ]:
# GETDEL — Get the value AND delete the key in one atomic operation
# Redis CLI: GETDEL otp:user1
# Perfect for one-time-use tokens!

r.set('otp:user1', '482931')
print(f"OTP before GETDEL: {r.get('otp:user1')}")

otp = r.getdel('otp:user1')  # Gets the value and deletes the key
print(f"GETDEL returned: {otp}")
print(f"OTP after GETDEL: {r.get('otp:user1')}")  # None — it's gone!

In [ ]:
# GETEX — Get the value AND set a new expiry
# Redis CLI: GETEX session:data EX 60
# Perfect for refreshing session timeouts!

r.set('session:data', 'user_session_info')
print(f"TTL before GETEX: {r.ttl('session:data')}")  # -1 (no expiry)

value = r.getex('session:data', ex=60)  # Get + set 60s expiry
print(f"Value: {value}")
print(f"TTL after GETEX: {r.ttl('session:data')}")  # ~60 seconds

---
## 3. APPEND and STRLEN

In [ ]:
# APPEND — Add to the end of a string
# Redis CLI: APPEND log "[INFO] Server started\n"

r.delete('log')
r.append('log', '[INFO] Server started\n')
r.append('log', '[INFO] User logged in\n')
r.append('log', '[WARN] High memory usage\n')

print("Log contents:")
print(r.get('log'))

# STRLEN — Get the length of the value
# Redis CLI: STRLEN log
print(f"Log size: {r.strlen('log')} characters")

---
## 4. Numeric Operations — Redis as a Calculator

Even though values are stored as strings, Redis can perform math on them! These operations are **atomic** — safe for concurrent access.

In [ ]:
# INCR / INCRBY / DECR / DECRBY
r.set('score', 100)

r.incr('score')             # +1 → 101
print(f"After INCR: {r.get('score')}")

r.incrby('score', 50)       # +50 → 151
print(f"After INCRBY 50: {r.get('score')}")

r.decr('score')             # -1 → 150
print(f"After DECR: {r.get('score')}")

r.decrby('score', 30)       # -30 → 120
print(f"After DECRBY 30: {r.get('score')}")

In [ ]:
# INCRBYFLOAT — For decimal numbers
# Redis CLI: SET price 9.99
r.set('price', '9.99')

# Redis CLI: INCRBYFLOAT price 2.50
r.incrbyfloat('price', 2.50)
print(f"After adding $2.50: ${r.get('price')}")

# Subtract by using negative value
r.incrbyfloat('price', -1.00)
print(f"After subtracting $1.00: ${r.get('price')}")

In [ ]:
# Cool trick: INCR on a non-existent key starts from 0!
r.delete('new_counter')

r.incr('new_counter')  # 0 + 1 = 1
r.incr('new_counter')  # 1 + 1 = 2
r.incr('new_counter')  # 2 + 1 = 3
print(f"Counter created by INCR alone: {r.get('new_counter')}")

---
## 5. SETRANGE / GETRANGE — Substring Operations

In [ ]:
# GETRANGE — Get a substring (like Python's string slicing)
# Redis CLI: GETRANGE greeting 0 4

r.set('greeting', 'Hello, World!')

print(f"Full string: {r.get('greeting')}")
print(f"[0:4]:  {r.getrange('greeting', 0, 4)}")    # 'Hello'
print(f"[7:11]: {r.getrange('greeting', 7, 11)}")   # 'World'
print(f"[-6:-1]: {r.getrange('greeting', -6, -1)}") # 'orld!'

# SETRANGE — Overwrite part of a string
# Redis CLI: SETRANGE greeting 7 "Redis"
r.setrange('greeting', 7, 'Redis!')
print(f"After SETRANGE: {r.get('greeting')}")  # 'Hello, Redis!'

---
## 6. Bit Operations (Brief Intro)

Redis lets you manipulate individual **bits** within a string. This is incredibly memory-efficient for certain use cases like tracking daily active users.

Imagine tracking whether each of 1 million users was active today. With bits, that's only **~125 KB** of memory!

In [ ]:
# Track daily active users — each user ID is a bit position
# Bit = 1 means "active", Bit = 0 means "not active"

# Redis CLI: SETBIT active:2024-01-15 1001 1
r.setbit('active:today', 1001, 1)  # User 1001 was active
r.setbit('active:today', 1002, 1)  # User 1002 was active
r.setbit('active:today', 1003, 0)  # User 1003 was NOT active (default)
r.setbit('active:today', 1004, 1)  # User 1004 was active

# Check if a specific user was active
# Redis CLI: GETBIT active:2024-01-15 1001
print(f"Was user 1001 active? {r.getbit('active:today', 1001)}")  # 1 = yes
print(f"Was user 1003 active? {r.getbit('active:today', 1003)}")  # 0 = no

# Count total active users
# Redis CLI: BITCOUNT active:2024-01-15
print(f"Total active users today: {r.bitcount('active:today')}")

---
## 7. Storing JSON in Strings

You can store complex objects by serializing them to JSON. This is useful when you want to cache entire API responses or objects.

In [ ]:
# Store a Python dictionary as JSON string
user_data = {
    'name': 'Sujit',
    'age': 25,
    'skills': ['Python', 'Redis', 'Docker'],
    'active': True
}

# Serialize to JSON and store
r.set('user:1001:json', json.dumps(user_data))
print(f"Stored as: {r.get('user:1001:json')}")

# Retrieve and deserialize
retrieved = json.loads(r.get('user:1001:json'))
print(f"\nRetrieved as Python dict:")
print(f"  Name: {retrieved['name']}")
print(f"  Skills: {retrieved['skills']}")
print(f"  Type: {type(retrieved)}")

> **JSON String vs Hash:** Use JSON strings when you always read/write the **entire** object. Use Hashes (Notebook 05) when you need to read/write **individual fields**. Hashes are more memory-efficient for objects.

---
## 8. Real-World Example: Simple Rate Limiter

**Problem:** Allow a user maximum 5 API requests per 60 seconds.

**Solution:** Use INCR + EXPIRE. Each user gets a counter that auto-resets every 60 seconds.

In [ ]:
def is_rate_limited(user_id, max_requests=5, window_seconds=60):
    """Check if a user has exceeded their rate limit."""
    key = f"ratelimit:{user_id}"
    
    # Increment the counter
    current = r.incr(key)
    
    # If this is the first request, set the expiry window
    if current == 1:
        r.expire(key, window_seconds)
    
    if current > max_requests:
        ttl = r.ttl(key)
        return True, f"Rate limited! Try again in {ttl} seconds."
    else:
        remaining = max_requests - current
        return False, f"Request allowed. {remaining} requests remaining."

# Simulate API requests from user "alice"
r.delete('ratelimit:alice')  # Reset for demo

for i in range(8):
    limited, message = is_rate_limited('alice', max_requests=5, window_seconds=60)
    status = "BLOCKED" if limited else "OK"
    print(f"  Request {i+1}: [{status}] {message}")

---
## 9. Real-World Example: Page View Counter

In [ ]:
def record_page_view(page_name):
    """Record a page view and return total views."""
    key = f"pageviews:{page_name}"
    total = r.incr(key)  # Atomic increment
    return total

def get_page_views(page_name):
    """Get total views for a page."""
    return r.get(f"pageviews:{page_name}") or '0'

# Simulate page views
pages = ['home', 'about', 'home', 'products', 'home', 'about', 'home']

for page in pages:
    views = record_page_view(page)
    print(f"  Visited /{page} — total views: {views}")

print("\n--- Page View Summary ---")
for page in ['home', 'about', 'products']:
    print(f"  /{page}: {get_page_views(page)} views")

---
## Cleanup

In [ ]:
r.flushdb()
print("Database cleaned up!")

---
## Key Takeaways

| Command | What It Does | Example Use Case |
|---|---|---|
| `SET key val EX 30` | Set with 30s expiry | Sessions, cache |
| `SET key val NX` | Set only if not exists | Distributed locks |
| `SET key val XX` | Set only if exists | Safe updates |
| `INCR / INCRBY` | Atomic increment | Counters, rate limiting |
| `INCRBYFLOAT` | Float increment | Prices, scores |
| `APPEND` | Append to string | Logs, building strings |
| `GETRANGE` | Get substring | Parsing stored data |
| `GETDEL` | Get and delete | One-time tokens |
| `GETEX` | Get and set expiry | Session refresh |
| `SETBIT / GETBIT` | Bit manipulation | User activity tracking |
| `BITCOUNT` | Count set bits | Active user counts |

### Key Insight
Strings are **way more** than just text. With INCR, bits, and expiry, they can power counters, locks, rate limiters, and analytics — all with atomic, thread-safe operations.

---
## Exercises

1. **Countdown Timer:** Create a key `timer` with value `10`. Decrement it in a loop until it reaches 0, printing each value. Use `time.sleep(0.5)` between decrements.

2. **One-Time Password:** Write a function `create_otp(user_id)` that generates a random 6-digit code, stores it with a 5-minute expiry, and returns the code. Write another function `verify_otp(user_id, code)` that checks the code using GETDEL (so it can only be used once).

3. **Visit Tracker with Bitmap:** Use SETBIT to track which of users 0-9 visited your site today. Set bits for users 0, 3, 5, 7. Then use BITCOUNT to count total visitors and GETBIT to check specific users.

4. **JSON Cache:** Write a function that "simulates" a slow database query (use `time.sleep(1)`). Cache the result as JSON in Redis with a 30-second TTL. On subsequent calls, return the cached version instantly. Print timing for both cached and uncached calls.

5. **Advanced Rate Limiter:** Modify the rate limiter to use a sliding window: track requests per second instead of a fixed window. (Hint: use the current second as part of the key name)

In [ ]:
# Your exercises here!


---
**Next up: [Notebook 03 — Lists](./03_Lists.ipynb)** — Ordered collections, queues, activity feeds, and the power of LPUSH/RPUSH!